# Inference

Loads the packaged model (`models/model.joblib`) exactly as the API does and scores patients. Because the file is the **whole** sklearn pipeline (imputation, scaling, one-hot encoding and classifier), raw patient fields go in and no separate preprocessing code is needed.

In [1]:
import json, sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import requests

from src import data
from src.predict import load_model, predict

## 1. Score a single patient with the library function

In [2]:
patient = {'age': 63, 'sex': 1, 'cp': 1, 'trestbps': 145, 'chol': 233, 'fbs': 1, 'restecg': 2,
           'thalach': 150, 'exang': 0, 'oldpeak': 2.3, 'slope': 3, 'ca': 0, 'thal': 6}
predict([patient])[0]

{'prediction': 0,
 'diagnosis': 'no heart disease',
 'confidence': 0.8345,
 'probability_disease': 0.1655}

## 2. Score a batch and compare with the recorded diagnosis

These rows come from the full dataset, so ~80% of them were seen in training; this demonstrates the plumbing, not generalisation (see the training notebook for held-out metrics).

In [3]:
df = data.load_clean().sample(8, random_state=3)
out = pd.DataFrame(predict(df[data.FEATURES].to_dict('records')), index=df.index)
out['actual'] = df['target'].map({0: 'no heart disease', 1: 'heart disease'})
out['correct'] = out['diagnosis'] == out['actual']
out

,prediction,diagnosis,confidence,probability_disease,actual,correct
245,1,heart disease,0.7080,0.7080,heart disease,True
162,0,no heart disease,0.9731,0.0269,no heart disease,True
10,1,heart disease,0.5085,0.5085,no heart disease,False
161,1,heart disease,0.7791,0.7791,heart disease,True
73,1,heart disease,0.7183,0.7183,heart disease,True
16,0,no heart disease,0.7498,0.2502,heart disease,False
180,1,heart disease,0.7801,0.7801,heart disease,True
195,1,heart disease,0.9446,0.9446,heart disease,True


## 3. Robustness: missing and unseen values

The pipeline imputes missing numerics/categoricals and ignores unseen categories instead of crashing.

In [4]:
odd = dict(patient, chol=None, thal=99)
predict([odd])[0]

{'prediction': 0,
 'diagnosis': 'no heart disease',
 'confidence': 0.7841,
 'probability_disease': 0.2159}

## 4. Same patient through the HTTP API

Requires the service on port 8000 (`make docker-run` or `docker compose up`).

In [5]:
try:
    r = requests.post('http://localhost:8000/predict', json=patient, timeout=5)
    print(r.status_code, json.dumps(r.json(), indent=2))
    assert r.json()['prediction'] == predict([patient])[0]['prediction'], 'API and library disagree'
    print('API result matches library result')
except requests.RequestException as exc:
    print('API not running - skipped:', exc)

200 {
  "prediction": 0,
  "diagnosis": "no heart disease",
  "confidence": 0.8345,
  "probability_disease": 0.1655
}
API result matches library result
